In [1]:
import torch

from tts.config.ndaligner.data_config import DataConfig
from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module


/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Path and Configs

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

aligner_training_module_cfg_path = "./runs/nd_aligner_vctk_v1.3_ablation_20260623-074803/model_config.json"
aligner_training_module_ckpt_path = "./runs/nd_aligner_vctk_v1.3_ablation_20260623-074803/checkpoints_timit_bae/last.pth"

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"

## INIT Models

In [ ]:
model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(config=model_config).to(device=device)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cpu.
Loading nested state_dict from key 'model' in ./runs/nd_aligner_vctk_v1.3_ablation_20260623-074803/checkpoints_timit_bae/last.pth
⚠️ Checkpoint load summary (strict=False):
  - Unexpected top-level modules: ['vocoder']
Checkpoint loading process finished.


## Init others

In [4]:
from tts.tokenizer.espeak_tokenizer import ESPEAKTokenizer
from tts.models.modules.spk_encoder import ECAPASpeakerEncoder

txt_tokenizer = ESPEAKTokenizer()
spk_encoder = ECAPASpeakerEncoder(device=device)

SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cuda.


## INIT BenchMarkers

In [5]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker
from tts.models.utils.input_maker import AlignerInputMaker

input_maker = AlignerInputMaker(
    audio_config=model_config.nd_aligner.audio,
    preprocess_config=model_config.nd_aligner.preprocess,
    tokenizer_type=model_config.nd_aligner.tokenizer_type,
    device="cuda"
)

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=input_maker,
    hyp_ignore_symbols=txt_tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cuda.
[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [32]:
with torch.no_grad():
    metrics = timit_benchmarker.__call__(
        aligner=aligner,
        vocoder=None,
        max_test_samples=100,
    )

Computing Alignments:   0%|          | 0/100 [00:00<?, ?it/s]

Computing Alignments: 100%|██████████| 100/100 [00:22<00:00,  4.40it/s]


In [33]:
print(metrics.word_boundary_error)
print(metrics.p_word_100ms)
print(metrics.p_word_50ms)
print(metrics.p_word_25ms)
print(metrics.p_word_10ms)

0.02463725581765175
97.41784334182739
89.31924700737
68.01643371582031
33.39201807975769
